# Hybrid Demucs

Feel free to use the Colab version:
https://colab.research.google.com/drive/1dC9nVxk3V_VPjUADsnFu8EiT-xnU1tGH?usp=sharing

In [1]:
!pip install -U demucs
# or for local development, if you have a clone of Demucs
# pip install -e .

     ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
     ---------------------------------------- 1.2/1.2 MB 12.2 MB/s  0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/6.2 MB ? eta -:--:--
   --------------------------- ------------ 4.2/6.2 MB 19.4 MB/s eta 0:00:01
   ------

  DEPRECATION: Building 'demucs' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'demucs'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'julius' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'julius'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'antlr4-python3-runtime' using the legacy setup.py bdist_wheel m

In [1]:
# You can use the `demucs` command line to separate tracks
!demucs test.mp3

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "D:\Anaconda\Scripts\demucs.exe\__main__.py", line 2, in <module>
  File "D:\Anaconda\Lib\site-packages\demucs\separate.py", line 12, in <module>
    from dora.log import fatal
  File "D:\Anaconda\Lib\site-packages\dora\__init__.py", line 66, in <module>
    from .explore import Explorer, Launcher
  File "D:\Anaconda\Lib\site-packages\dora\explore.py", line 27, in <module>
    from .shep import Shepherd, Sheep
  File "D:\Anaconda\Lib\site-packages\dora\shep.py", line 25, in <module>
    from .distrib import get_distrib_spec
  File "D:\Anaconda\Lib\site-packages\dora\distrib.py", line 14, in <module>
    import torch
  File "D:\Anaconda\Lib\site-packages\torch\__init__.py", line 262, in <module>
    _load_dll_libraries()
  File "D:\Anaconda\Lib\site-packages\torch\__init__.py", line 245, in _load_dll_libraries
    raise err
OSError: [WinError

In [3]:
# You can also load directly the pretrained models,
# for instance for the MDX 2021 winning model of Track A:
from demucs import pretrained
model = pretrained.get_model('mdx')

OSError: [WinError 127] The specified procedure could not be found. Error loading "d:\Anaconda\Lib\site-packages\torch\lib\c10_cuda.dll" or one of its dependencies.

In [ ]:
# Because `model` is a bag of 4 models, you cannot directly call it on your data,
# but the `apply_model` will know what to do of it.
import torch
from demucs.apply import apply_model
x = torch.randn(1, 2, 44100 * 10)  # ten seconds of white noise for the demo
out = apply_model(model, x)[0]     # shape is [S, C, T] with S the number of sources

# So let see, where is all the white noise content is going ?
for name, source in zip(model.sources, out):
    print(name, source.std() / x.std())
# The outputs are quite weird to be fair, not what I would have expected.

In [ ]:
# now let's take a single model from the bag, and let's test it on a pure cosine
freq = 440  # in Hz
sr = model.samplerate
t = torch.arange(10 * sr).float() / sr
x = torch.cos(2 * 3.1416 * freq * t).expand(1, 2, -1)
sub_model = model.models[3]
out = sub_model(x)[0]

# Same question where does it go?
for name, source in zip(model.sources, out):
    print(name, source.std() / x.std())
    
# Well now it makes much more sense, all the energy is going
# in the `other` source.
# Feel free to try lower pitch (try 80 Hz) to see what happens !

In [ ]:
# For training or more fun, refer to the Demucs README on our repo
# https://github.com/facebookresearch/demucs/tree/main/demucs